# 06 · Full API Contract Smoke Suite
One-shot health check across **every** endpoint group. Useful as a pre-release
validation or post-deploy canary confirmation.

Run all cells sequentially. Each cell prints `PASS` or `FAIL` with status code.
A summary matrix is emitted at the end.

In [ ]:
import json, urllib.request, urllib.error
API_BASE = 'https://api.banana.engineer'

def call_endpoint(method, path, payload=None):
    url = f"{API_BASE}{path}"
    h = {'Accept': 'application/json'}
    data = None
    if payload is not None:
        h['Content-Type'] = 'application/json'
        data = json.dumps(payload).encode()
    req = urllib.request.Request(url, data=data, headers=h, method=method.upper())
    try:
        with urllib.request.urlopen(req) as r:
            return {'ok': True, 'status': r.status, 'body': r.read().decode('utf-8', errors='replace')}
    except urllib.error.HTTPError as e:
        return {'ok': False, 'status': e.code, 'body': e.read().decode('utf-8', errors='replace')}
    except Exception as ex:
        return {'ok': False, 'status': None, 'body': str(ex)}

def get(path):  return call_endpoint('GET',  path)
def post(path, payload=None): return call_endpoint('POST', path, payload)

RESULTS = []  # accumulated smoke results

def smoke(label, method, path, payload=None, expect_ok=True):
    r = call_endpoint(method, path, payload)
    verdict = 'PASS' if (r['ok'] == expect_ok or r['status'] in (200,201,204)) else 'FAIL'
    RESULTS.append({'label': label, 'method': method, 'path': path, 'status': r['status'], 'verdict': verdict})
    print(f"[{verdict}] {method:6} {path:50}  HTTP {r['status']}")
    return r

print('smoke_harness_ready')

In [ ]:
# ── Group: Health ─────────────────────────────────────────────────────────
smoke('health',          'GET', '/health')

In [ ]:
# ── Group: Banana ─────────────────────────────────────────────────────────
smoke('banana-summary',  'GET',  '/banana/summary')
smoke('banana-classify', 'POST', '/banana', {'purchases': 2, 'multiplier': 2})

In [ ]:
# ── Group: Not-Banana ─────────────────────────────────────────────────────
smoke('not-banana-summary', 'GET', '/not-banana/summary')

In [ ]:
# ── Group: Ripeness ───────────────────────────────────────────────────────
smoke('ripeness-summary',  'GET',  '/ripeness/summary')
smoke('ripeness-score',    'POST', '/ripeness', {'purchases': 3, 'multiplier': 2})

In [ ]:
# ── Group: ML / Model Ops ─────────────────────────────────────────────────
smoke('ml-training-history', 'GET', '/banana-ml/training/history')
smoke('ml-candidate',        'GET', '/banana-ml/candidate')

In [ ]:
# ── Group: Logistics ──────────────────────────────────────────────────────
smoke('trucks-list',   'GET', '/trucks')
smoke('harvest-list',  'GET', '/harvest')

In [ ]:
# ── Group: Chat & Streaming ───────────────────────────────────────────────
smoke('chat-list',         'GET', '/chat')
smoke('streaming-status',  'GET', '/streaming/status')

In [ ]:
# ── Group: Audit & Telemetry ──────────────────────────────────────────────
smoke('audit-log',   'GET', '/audit')
smoke('telemetry',   'GET', '/telemetry')

In [ ]:
# ── Group: OpenAPI Contract ───────────────────────────────────────────────
smoke('openapi-spec', 'GET', '/swagger/v1/swagger.json')

In [ ]:
# ── Summary matrix ────────────────────────────────────────────────────────
passed = [r for r in RESULTS if r['verdict'] == 'PASS']
failed = [r for r in RESULTS if r['verdict'] == 'FAIL']

print(f'\n══ Smoke Results: {len(passed)}/{len(RESULTS)} PASS ══')
if failed:
    print('\nFAILED:')
    for row in failed:
        print(f"  {row['method']:6} {row['path']}  HTTP {row['status']}")
else:
    print('All checks passed.')